<div class="alert alert-info">

<h3>Задание (выполнять в отдельном файле)</h3>
<p></p>
Обучить нейронную сеть классифицировать ирисы. Для этого:
<ol>
<li>Предварительно разбейте выборку на обучение и контроль с помощью train_test_split из sklearn.</li>
<li>Масштабируйте признаки с помощью sklearn.preprocessing.StandardScaler (fit_transform для обучающей выборки, transform для контрольной).</li>
<li>Преобразуйте данные в тензоры нужного типа.</li>
<li>Реализуйте архитектуру нейронной сети, приведенную на рисунке ниже.</li>
<li>Обучите данную архитектуру. Для обучения используйте метод torch.optim.Adam с шагом обучения lr=0.01. Количество эпох – не менее 100.</li>
<li>Получите предсказания модели на контрольной выборке и оцените accuracy_score полученной сети.</li>
</ol>    
 <p></p>
</div>

<img src="img/Fig6.PNG" alt="Image 1" style="width: 350px;">

In [35]:
from sklearn.datasets import load_iris

In [36]:
iris_data = load_iris()
X = iris_data.data
y = iris_data.target

In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [38]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [39]:
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.LongTensor(y_train)

X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.LongTensor(y_test)

In [40]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size=4, num_classes=3):
        super(NeuralNetwork, self).__init__()
        
        self.input_layer = nn.Linear(in_features=input_size, out_features=10)
        self.hidden_layer = nn.Linear(in_features=10, out_features=10)
        self.output_layer = nn.Linear(in_features=10, out_features=num_classes)
        self.activation = nn.Sigmoid() 
        
    def forward(self, inputs):
        u = self.activation(self.input_layer(inputs))
        u = self.activation(self.hidden_layer(u))
        u = self.output_layer(u)
        return u

In [41]:
model = NeuralNetwork(input_size=4, num_classes=3)
model

NeuralNetwork(
  (input_layer): Linear(in_features=4, out_features=10, bias=True)
  (hidden_layer): Linear(in_features=10, out_features=10, bias=True)
  (output_layer): Linear(in_features=10, out_features=3, bias=True)
  (activation): Sigmoid()
)

In [43]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
n_epochs = 100+1
train_losses = []

for i in range(n_epochs):
    optimizer.zero_grad()
    pred = model(X_train_tensor)
    loss = criterion(pred, y_train_tensor)
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    if i % 10 == 0:
        print(f'Эпоха: {i} Потеря: {loss.item()}')

Эпоха: 0 Потеря: 0.19909970462322235
Эпоха: 10 Потеря: 0.14729586243629456
Эпоха: 20 Потеря: 0.11149442195892334
Эпоха: 30 Потеря: 0.08899665623903275
Эпоха: 40 Потеря: 0.07513702660799026
Эпоха: 50 Потеря: 0.0662941113114357
Эпоха: 60 Потеря: 0.06042921170592308
Эпоха: 70 Потеря: 0.05643483251333237
Эпоха: 80 Потеря: 0.05363410711288452
Эпоха: 90 Потеря: 0.0516137033700943
Эпоха: 100 Потеря: 0.050114624202251434


In [44]:
model.eval()

with torch.no_grad():
    pred_test = model(X_test_tensor)
    predicted_classes = torch.argmax(pred_test, dim=1)

In [45]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test_tensor.numpy(), predicted_classes.numpy())

1.0